In [ ]:
import ROOT

# RDataFrame

## ROOT RDataFrame - recap

[RDataFrame documentation](https://root.cern/doc/master/classROOT_1_1RDataFrame.html)

- RDF is ROOT's high-level analysis interface, it is built with a modular and flexible workflow in mind

- Users define their analysis as a sequence of operations to be performed on the data-frame object; 

    - the framework takes care of the management of the loop over entries as well as low-level details such as I/O and parallelisation.

- RDataFrame provides methods to perform most common operations required by ROOT analyses: 

    - at the same time, users can just as easily specify custom code that will be executed in the event loop.

<img src="../images/rdf_1.png" style="width: 100%;">

### Important Reminder!
Make sure to **book all transformations and actions before** you access the contents of any of the results: this lets RDataFrame accumulate work and then produce all results at the same time, upon first access to any of them.

In [ ]:
treename = "dataset"
filename = "../data/example_file.root"

df_wrong = ROOT.RDataFrame(treename, filename)

h_a = df_wrong.Histo1D("a")
h_a_val = h_a.GetValue()

h_b = df_wrong.Histo1D("b")
h_b_val = h_b.GetValue()

h_vec1 = df_wrong.Histo1D("vec1")
h_vec1_val = h_vec1.GetValue()

print(f"The dataset was processed {df_wrong.GetNRuns()} times.")


In [ ]:
df_good = ROOT.RDataFrame(treename, filename)

h_a = df_good.Histo1D("a")
h_b = df_good.Histo1D("b")
h_vec1 = df_good.Histo1D("vec1")

h_a_val = h_a.GetValue()
h_b_val = h_b.GetValue()
h_vec1_val = h_vec1.GetValue()

print(f"The dataset was processed {df_good.GetNRuns()} time.")

# 1. Collections

## Working with `numpy` arrays
- RDataFrame offers interoperability with `numpy` arrays. 

- It can be created from a dictionary of such arrays and it can also export its contents to the same format. 

- All operations are available also when using the `numpy`- based dataset.

- **Note:** this support is limited to one-dimensional numpy arrays, which are directly mapped to columns in the RDataFrame.

In [ ]:
import numpy

np_dict = {colname: numpy.random.rand(100) for colname in ["a","b","c"]}

df = ROOT.RDF.FromNumpy(np_dict)

print(f"Columns in the RDataFrame: {df.GetColumnNames()}")

In [ ]:
co = df.Count()
m_a = df.Mean("a")

fil1 = df.Filter("c < 0.7")
def1 = fil1.Define("d", "a+b+c")
h = def1.Histo1D("d")

c = ROOT.TCanvas()
h.Draw()

print(f"Number of rows in the dataset: {co.GetValue()}")
print(f"Average value of column a: {m_a.GetValue()}")
c.Draw()

In [ ]:
# Export the modified dataframe to a dictionary of numpy arrays

np_dict_mod = def1.AsNumpy()

np_dict_mod

## Working with `pandas` dataframe

As a convenience, RDataFrame can also ingest data from a [`pandas`](https://pandas.pydata.org/) dataframe.

In [ ]:
import pandas as pd

# Let's create some data in a pandas dataframe
pdf = pd.DataFrame({'x': [1, 2, 3], 'y': [4, 5, 6]})

# Convert the Pandas DataFrame to RDataFrame
# The column names are directly copied to the RDF 
# Please note that only fundamental types (int, float, ...) are supported and
# the arrays must have the same length.
df = ROOT.RDF.FromPandas(pdf)

# You can now use the RDataFrame as usual, e.g. add a column ...
df = df.Define('z', 'x + y')

# ... or print the content
df.Display().Print()

## Working with `awkward` arrays

Utility functions are available for interoperability with the `awkward` library, described at https://awkward-array.org/doc/main/user-guide/how-to-convert-rdataframe.html.

The function for RDataFrame to Awkward conversion is ak.from_rdataframe(). The argument to this function accepts a tuple of strings that are the RDataFrame column names. By default this function returns ak.Array type.

In [ ]:
import awkward as ak

treename = "myDataset"
filename = "../data/collections_dataset.root"

df = ROOT.RDataFrame(treename, filename)

array = ak.from_rdataframe(
    df,
    columns=("E", "nPart", "px", "py")
)

array

The function for Awkward to RDataFrame conversion is ak.to_rdataframe().

The argument to this function requires a dictionary: { `<column name string>` : `<awkward array>` }. This function always returns an RDataFrame object.

The arrays given for each column have to be equal length:

In [ ]:
array_x = ak.Array(
    [
        {"x": [1.1, 1.2, 1.3]},
        {"x": [2.1, 2.2]},
        {"x": [3.1]},
        {"x": [4.1, 4.2, 4.3, 4.4]},
        {"x": [5.1]},
    ]
)
array_y = ak.Array([1, 2, 3, 4, 5])
array_z = ak.Array([[1.1], [2.1, 2.3, 2.4], [3.1], [4.1, 4.2, 4.3], [5.1]])

assert len(array_x) == len(array_y) == len(array_z)

df = ak.to_rdataframe({"x": array_x, "y": array_y, "z": array_z})

print(f"Columns and their types: {({str(col): df.GetColumnType(col) for col in df.GetColumnNames()})}")
df.Display(["x","y","z"]).Print()

## Recap from the student course: Collections and object selections

- RDataFrame reads collections as the special type [ROOT::RVec](https://root.cern/doc/master/classROOT_1_1VecOps_1_1RVec.html) - e.g. a branch containing an array of floating point numbers can be read as a `ROOT::RVec<float>`.

- C-style arrays (with variable or static size), `std::vectors` and many other collection types can be read this way. 

- When reading ROOT data, column values of type `ROOT::RVec<T>` perform no copy of the underlying array.

- `RVec` is a container similar to `std::vector` (and can be used just like a `std::vector`) but it also offers a rich interface to operate on the array elements in a vectorised fashion, similarly to Python's NumPy arrays.

In [ ]:
treename = "myDataset"
filename = "../data/collections_dataset.root"
df = ROOT.RDataFrame(treename, filename)

print(f"Columns in the dataset: {df.GetColumnNames()}")

To quickly inspect the data we can export it as a dictionary of `numpy` arrays thanks to the `AsNumpy` RDataFrame method. 

Note that for each row, `E` is an array of values:

In [ ]:
npy_dict = df.AsNumpy(["E"])

for row, vec in enumerate(npy_dict["E"]):
    print(f"\nRow {row} contains:\n{vec}")

### Define a new column with operations on RVecs

In [ ]:
df1 = df.Define("good_pt", "sqrt(px*px + py*py)[E>100]")

`sqrt(px*px + py*py)[E>100]`:
- `px`, `py` and `E` are the columns, the elements of those columns are `RVec`s

- Operations on `RVec`s, such as sum, product, sqrt, preserve the dimensionality of the array

- `[E>100]` selects the elements of the array that satisfy the condition

- `E > 100`: boolean expressions on `RVec`s such as `E > 100` return a mask, that is an array with information which values pass the selection (e.g. `[0, 1, 0, 0]` if only the second element satisfies the condition)

### Now we can plot the newly defined column values in a histogram

In [ ]:
c = ROOT.TCanvas()
h = df1.Histo1D(("pt", "pt", 16, 0, 4), "good_pt")
h.Draw()
c.Draw()

# 2. RDatsetSpec and FromSpec

What if you have many different samples - multiple TTree or RNTuple names and many different file names with some metadata information as well? 
- We have the RDatasetSpec class for that!
- And especially, get familiar with the `FromSpec` method thanks to which you can create a single dataframe consisting of many different samples which are added via a JSON file.

Example:
```json
{
   "samples": {
      "sampleA": {
         "trees": ["tree1", "tree2"],
         "files": ["file1.root", "file2.root"],
         "metadata": {"lumi": 1.0, }
      },
      "sampleB": {
         "trees": ["tree3", "tree4"],
         "files": ["file3.root", "file4.root"],
         "metadata": {"lumi": 0.5, }
      },
      ...
    },
}
```

In [ ]:
df = ROOT.RDF.Experimental.FromSpec("../data/dataset_spec.json")

And now simply continue your analysis as per usual... 

But what if some operations are valid only for some of the samples? 
- Solution: use `DefinePerSample` method - the samples can be recognized thanks to one or more metadata instances and then we can make use of `RSampleInfo` class. 

In [ ]:
df = df.DefinePerSample("xsecs", 'rdfsampleinfo_.GetD("xsecs")') 
# This is a teaser - how this can be used further is shown in the extra chapter

# 3. Dealing with missing values in the input dataset

Often times, the input dataset is a coherent set of full entries. That is, given a dataset schema with columns `("x", "y", "z")`, all files in the chain always contain those columns and every entry will be whole. This can be extended to horizontal compositions, called "friends" in the TTree jargon, where the schema of the main dataset can be extended with the columns of the friend(s) dataset(s).

In some cases, entries retrieved in the event loop might be not completely filled. Instead they might present some missing values. In this notebook we explore two notable scenarios where this might happen.

### Scenario 1: Some columns are missing at one or more files in the chain

When processing a dataset made of multiple files, the schema is implicitly assumed to be the one of the first file opened. In some cases, a column present in the first file might not be present in other files, and vice versa. This could happen for example when processing different samples (e.g. data and simulation).

In the next cell, we create a dataset made of three files. The first file contains both columns `x` and `y`, the second file only contains column `y`, the third file only contains column `x`. In practice this would lead to a dataset looking like:

| File # | x | y  |
|--------|---|----|
| 1      | 1 | 11 |
| 1      | 2 | 22 |
| 1      | 3 | 33 |
| 2      |   | 44 |
| 2      |   | 55 |
| 2      |   | 66 |
| 3      | 7 |    |
| 3      | 8 |    |
| 3      | 9 |    |

In [ ]:
from missing_values_utils import DatasetMissingBranches
DatasetMissingBranches.create_dataset()

We create the input dataset for the RDataFrame by chaining the three files together in a TChain object. Operations in the event loop **expect the value to be there** to function properly, so RDataFrame offers some facilities to decide what to do in case of a missing value:

* `DefaultValueFor(colname, defaultval)`: lets the user provide one default value for the current entry of the input column, in case the value is missing.
* `FilterAvailable(colname)`: works in the same way as the traditional `Filter` operation, where the "expression" is "is the value available?". If so, the entry is kept, if not, it is discarded.
* `FilterMissing(colname)`: works in the same way as the traditional `Filter` operation, where the "expression" is "is the value missing?". If so, the entry is kept, if not, it is discarded.

In [ ]:
import ROOT

chain = ROOT.TChain()
for fname, tname in zip(DatasetMissingBranches.filenames, DatasetMissingBranches.treenames):
    chain.Add(fname + "?#" + tname)

df = ROOT.RDataFrame(chain)

If we want to get a complete view of all columns and all entries, we must make sure that there is always a value to display, so we use `DefaultValueFor` on both columns.

In [ ]:
default_value = ROOT.std.numeric_limits[int].min()

# Example 1: provide a default value for all missing branches
display_1 = (
    df.DefaultValueFor("x", default_value)
    .DefaultValueFor("y", default_value)
    .Display(columnList=("x", "y"), nRows=15)
)
display_1.Print()

If instead we want to use the information that some value is missing, we can for example skip the events where values of column `x` are missing with `FilterAvailable`.

In [ ]:
# Example 2: provide a default value for branch y, but skip events where
# branch x is missing
display_2 = (
    df.DefaultValueFor("y", default_value)
    .FilterAvailable("x")
    .Display(columnList=("x", "y"), nRows=15)
)
display_2.Print()

Finally, we could decide to keep all the events where the column `y` is missing with `FilterMissing`.

In [ ]:
# Example 3: only keep events where branch y is missing and display values for branch x
display_3 = df.FilterMissing("y").Display(columnList=("x",), nRows=15)
display_3.Print()

In [ ]:
DatasetMissingBranches.cleanup_dataset()

### Scenario 2: Some values are missing due to a mismatch when joining with another dataset

The second scenario can happen with unaligned horizontal compositions, i.e. when joining the main dataset with one or more auxiliary datasets and the event order is not guaranteed across them. In case the current event being processed does not match one (or more) of the friend datasets, that would lead to a missing value in the friend column(s).

Let's take for example a dataset with one main file and two auxiliary files. The column `idx` is present in all files and is used to build an index that is then probed during the join step. The main file also has column `x`, the first auxiliary file also has column `y`, the second auxiliary file also has column `z`. Some values for the column `idx` in the main file are not present in the auxiliary files, so that after joining the full view on the dataset would be:

| idx | x | y | z |
|-----|---|---|---|
| 1   | 1 | 4 | 6 |
| 2   | 2 | 5 |   |
| 3   | 3 |   | 7 |

In [ ]:
from missing_values_utils import DatasetMismatchedJoin
DatasetMismatchedJoin.create_dataset()

Now we create the input dataset for the RDataFrame. The indexes for the two auxiliary datasets are built via the `BuildIndex` method. Then the auxiliary datasets are joined with the main one via the `AddFriend` method of the main dataset.

In [ ]:

main_chain = ROOT.TChain(DatasetMismatchedJoin.main_tree_name)
main_chain.Add(DatasetMismatchedJoin.main_file)

aux_chain_1 = ROOT.TChain(DatasetMismatchedJoin.aux_tree_name_1)
aux_chain_1.Add(DatasetMismatchedJoin.aux_file_1)
aux_chain_1.BuildIndex("idx")

aux_chain_2 = ROOT.TChain(DatasetMismatchedJoin.aux_tree_name_2)
aux_chain_2.Add(DatasetMismatchedJoin.aux_file_2)
aux_chain_2.BuildIndex("idx")

main_chain.AddFriend(aux_chain_1)
main_chain.AddFriend(aux_chain_2)

df = ROOT.RDataFrame(main_chain)

As above, we can decide to get a dataset with whole entries by providing a default value in case there is a mismatch in the join and thus a missing value.

In [ ]:
aux_tree_1_colidx = DatasetMismatchedJoin.aux_tree_name_1 + ".idx"
aux_tree_1_coly = DatasetMismatchedJoin.aux_tree_name_1 + ".y"
aux_tree_2_colidx = DatasetMismatchedJoin.aux_tree_name_2 + ".idx"
aux_tree_2_colz = DatasetMismatchedJoin.aux_tree_name_2 + ".z"

default_value = ROOT.std.numeric_limits[int].min()

# Example 1: provide default values for all columns in case there was no
# match
display_1 = (
    df.DefaultValueFor(aux_tree_1_colidx, default_value)
    .DefaultValueFor(aux_tree_1_coly, default_value)
    .DefaultValueFor(aux_tree_2_colidx, default_value)
    .DefaultValueFor(aux_tree_2_colz, default_value)
    .Display(
        ("idx", aux_tree_1_colidx, aux_tree_2_colidx, "x", aux_tree_1_coly, aux_tree_2_colz))
)
display_1.Print()

In this scenario, we can also obtain a fine-grained processing of the matched datasets by composing operations on the first auxiliary dataset and operations on the second one. For example, we can decide to skip the entries where there is no match in the first auxiliary tree but fill with default values the entries in the second auxiliary tree in case there is still some mismatch.

**Note**: The `FilterAvailable` operation is a filter, so will take precedence w.r.t. running the `DefaultValueFor` in the event loop.

In [ ]:

# Example 2: skip the entire entry when there was no match for a column
# in the first auxiliary tree, but keep the entries when there is no match
# in the second auxiliary tree and provide a default value for those
display_2 = (
    df.DefaultValueFor(aux_tree_2_colidx, default_value)
    .DefaultValueFor(aux_tree_2_colz, default_value)
    .FilterAvailable(aux_tree_1_coly)
    .Display(
            ("idx", aux_tree_1_colidx, aux_tree_2_colidx, "x", aux_tree_1_coly, aux_tree_2_colz))
)
display_2.Print()

And finally, we can also use `FilterMissing` to keep entries that would be empty due to a mismatch in the join.

In [ ]:
# Example 3: Keep entries from the main tree for which there is no
# corresponding match in entries of the first auxiliary tree
display_3 = df.FilterMissing(aux_tree_1_colidx).Display(("idx", "x"))
display_3.Print()

In [ ]:
DatasetMismatchedJoin.cleanup_dataset()

# 4. Systematic variations in the analysis

At some point in the development of a HEP data analysis workflow, the treatment of systematic variations in the observed quantities will become necessary. From the standpoint of a HEP physicist, the study of systematic variations involves many different, often conceptually complex cases. From the standpoint of the pure numerical computation, however, what typically happens is that the application must produce multiple results instead of a single one, each computed in a "universe" in which certain inputs take modified values.

In the next few cells, we explore the RDataFrame API devoted to helping the user include systematic variations in their analysis code.

In [ ]:
treename = "myDataset"
filename = "../data/collections_dataset.root"
df = ROOT.RDataFrame(treename, filename)

### Registering variations for one observable

As a basic example, let's see how to include two variations for the column `px`, including giving them special labels. Note that in the call to `Vary` we can use values from columns available in the dataset, including the column we are currently registering variations for. In this example, we are booking the nominal histogram via the usual `Histo1D` method.

Note the following in the example:

* Custom names for variations can be passed in a list via the `variationTags` parameter.
* The name of the input column is used as the default name for the variation, unless the `variationName` parameter is used as in this example.
* The full variation name will be composed of the varied column name and the variation tags (e.g. "mypt:down", "mypt:up" in this example).
* The histogram is filled with values of the `good_pt` column, defined after the `Vary` call. The presence of systematic variations for certain columns is automatically propagated through filters, defines and actions, and RDataFrame will take these dependencies into account when producing varied results.

In [ ]:
df = df.Define("pt", "sqrt(px*px + py*py)")

nominal_histo = (
    df.Vary(
        colName="pt",
        expression="ROOT::RVec<ROOT::RVecD>{pt*0.95, pt*1.05}",
        variationTags=["down", "up"],
        variationName="mypt")
      .Define("good_pt", "pt[E>100]")
      .Histo1D("good_pt")
)

In order to retrieve also all the varied histograms, we pass the pointer to the action just booked to the `VariationsFor` function, as shown below. This will return a dictionary containing the nominal histogram as well as all the varied histograms.

In [ ]:
all_histos = ROOT.RDF.Experimental.VariationsFor(nominal_histo)

c = ROOT.TCanvas()

all_histos["nominal"].SetLineColor(ROOT.kBlue)
all_histos["nominal"].Draw()

all_histos["mypt:down"].SetLineColor(ROOT.kRed)
all_histos["mypt:down"].Draw("SAME")

all_histos["mypt:up"].SetLineColor(ROOT.kGreen)
all_histos["mypt:up"].Draw("SAME")

c.Draw()

### Extra - registering variations for multiple columns simultaneously

The `Vary` function also allows to vary multiple columns simultaneously (in "lockstep"). The expression in this case must return an RVec of RVecs, one per column: each inner vector contains the varied values for one column, and the inner vectors follow the same ordering as the column names passed as first argument. Besides the variation tags, in this case we also have to explicitly pass a variation name as there is no one column name that can be used as default.

In [ ]:
df = ROOT.RDataFrame(treename, filename)

nominal_histo_lockstep = (
    df.Vary(
        colNames=["px", "py"],
        expression="ROOT::RVec<ROOT::RVec<ROOT::RVecD>>{{px*0.95, px*1.05}, {py*0.95, py*1.05}}",
        variationTags=["down", "up"],
        variationName="pxAndpy")
      .Define("pt_lockstep", "sqrt(px*px + py*py)[E>100]")
      .Histo1D("pt_lockstep")
)

In [ ]:
all_histos = ROOT.RDF.Experimental.VariationsFor(nominal_histo_lockstep)

c = ROOT.TCanvas()

all_histos["nominal"].SetLineColor(ROOT.kBlue)
all_histos["nominal"].Draw()

all_histos["pxAndpy:down"].SetLineColor(ROOT.kRed)
all_histos["pxAndpy:down"].Draw("SAME")

all_histos["pxAndpy:up"].SetLineColor(ROOT.kGreen)
all_histos["pxAndpy:up"].Draw("SAME")

c.Draw()

## 4.1 *Experimental*: Snapshot systematic variations to files

Customarily, snapshots only include nominal columns, that is, input columns or columns obtained via `Define`. In ROOT v6.38, RDataFrame acquired the ability to save systematic variations (see [details](https://root.cern/doc/master/classROOT_1_1RDataFrame.html#systematics)).
[`Snapshot`](https://root.cern/doc/master/classROOT_1_1RDF_1_1RInterface.html#a28923a701aac4a1218d39c1080e57bc3) can be customised using RDF's [`RSnapshotOptions`](https://root.cern/doc/master/structROOT_1_1RDF_1_1RSnapshotOptions.html). It can be used to include systematic variations in snapshots.

Let's prepare snapshot options to include variations:

In [ ]:
snap_options_vars = ROOT.RDF.RSnapshotOptions()
snap_options_vars.fIncludeVariations = True # Enable systematic variations
snap_options_vars.fOverwriteIfExists = True # So we can rerun the snapshot

{key: getattr(snap_options_vars, key) for key in dir(snap_options_vars) if key.startswith('f')}

Passing the snapshot options as the last argument to Snapshot:

In [ ]:
treename_output = "SnapshotDemo"
filename_output = "snapshotOutput.root"

df = ROOT.RDataFrame(10)
snapshot = (
    df.Define("eventNo", "rdfentry_")
    .Define("x", "1.f * rdfentry_")
    .Vary(
        colName="x",
        expression="ROOT::RVecF{x - 0.5f, x + 0.5f}",
        variationTags=["down", "up"],
        variationName="xVar")
    .Define("y", "-1*x")
    .Filter("x > 2 && y >= -8")
    .Snapshot(treename_output, filename_output, ["eventNo", "x", "y"], snap_options_vars)
)

Inspecting the result on the command line:

In [ ]:
%%bash
echo "The output file contains:"
rootls -l snapshotOutput.root

The output tree now contains the nominal columns and their variations. When filters are in use, the following can happen:
- All variations pass, so all columns are written.
- None of the variations pass. The event is omitted from the output.
- The mixed case: Some variations don't pass the filter. The values in these columns are set to zero to save space. This is the case e.g. for events 2 and 8 in the table below.

In [ ]:
with ROOT.TFile.Open(filename_output) as file:
    tree = file.Get(treename_output)
    tree.Scan()

### Reading snapshots with variations in RDataFrame: Dealing with missing values

When reading outputs resulting from a snapshot with variations with RDataFrame, one might read columns that don't have an entry for the specific event. RDataFrame uses a bitmask to mask these entries, to ensure that invalid columns are not read inadvertently. Trying to read e.g. the `"x"` column will result in an error:

In [ ]:
ROOT.RDataFrame(treename_output, filename_output).Display("x").GetValue()

To read these columns, one has to handle the missing values as shown in section 3, e.g. `FilterAvailable` or `DefaultValueFor`:

In [ ]:
display = (
    ROOT.RDataFrame(treename_output, filename_output)
    .FilterAvailable("x")
    .Display(columnList=("x", "y"))
)
display.GetValue()

Or reading all entries where the "Up" variation is valid, padding invalid values from the "down" variation with `1234.5`:

In [ ]:
display = (
    ROOT.RDataFrame(treename_output, filename_output)
    .FilterAvailable("x__xVar_up")
    .DefaultValueFor["float"]("x__xVar_down", 1234.5)
    .DefaultValueFor["float"]("y__xVar_down", -1234.5)
    .Display(columnList=("x__xVar_up", "y__xVar_up", "x__xVar_down", "y__xVar_down"))
)
display.GetValue()

# 5. Scaling up the analysis - multi-threading and distributed RDataFrame

## Using all cores of your machine with multi-threaded RDataFrame
- RDataFrame can transparently perform multi-threaded event loops to speed up the execution of its actions. 

- Users have to call `ROOT::EnableImplicitMT()` before constructing the RDataFrame object to indicate that it should take advantage of a pool of worker threads. 

- Each worker thread processes a distinct subset of entries, and their partial results are merged before returning the final values to the user.

- RDataFrame operations such as Histo1D or Snapshot are guaranteed to work correctly in multi-thread event loops. 

- User-defined expressions, such as strings or lambdas passed to `Filter`, `Define`, `Foreach`, `Reduce` or `Aggregate` will have to be thread-safe, i.e. it should be possible to call them concurrently from different threads.

In [ ]:
%%time
# Activate multithreading capabilities
# By default takes all available cores on the machine
ROOT.EnableImplicitMT()

# treename = "Events"
# filename = "root://eospublic.cern.ch//eos/opendata/cms/derived-data/AOD2NanoAODOutreachTool/Run2012BC_DoubleMuParked_Muons.root"
# df = ROOT.RDataFrame(treename, filename)

# df.Sum("nMuon").GetValue()

# Disable implicit multithreading when done
ROOT.DisableImplicitMT()

## Multiple concurrent RDataFrame runs
If your analysis needs multiple RDataFrames to run (for example multiple dataset samples, data vs simulation etc.), make use of `ROOT.RDF.RunGraphs` 

In [ ]:
ROOT.EnableImplicitMT()
treename1 = "myDataset"
filename1 = "../data/collections_dataset.root"
treename2 = "dataset"
filename2 = "../data/example_file.root"

df1 = ROOT.RDataFrame(treename1, filename1)
df2 = ROOT.RDataFrame(treename2, filename2)
h1 = df1.Histo1D("px")
h2 = df2.Histo1D("a")


ROOT.RDF.RunGraphs((h1, h2))
ROOT.DisableImplicitMT()

In [ ]:
c = ROOT.TCanvas()
h1.Draw()
c.Draw()

In [ ]:
c = ROOT.TCanvas()
h2.Draw()
c.Draw()

## Distributed RDataFrame

An `RDataFrame` analysis written in Python can be executed both *locally* - possibly in parallel on the cores of the machine - and *distributedly* by offloading computations to external resources, which include:

- [Spark](https://spark.apache.org/) and 
- [Dask](https://dask.org/) clusters. 

- This feature is enabled by the architecture depicted below.

- It shows that RDataFrame computation graphs can be mapped to different kinds of resources via backends.

- In this notebook we will exercise the Dask backend, which divides an `RDataFrame` input dataset in logical ranges and submits computations for each of those ranges to Dask resources.

<img src="../images/DistRDF_architecture.png" alt="Distributed RDataFrame">

### Create a Dask client

- In order to work with a Dask cluster we need a `Client` object.
- It represents the connection to that cluster and allows to configure execution-related parameters (e.g. number of cores, memory). 
- The client object is just the intermediary between our client session and the cluster resources, 
- Dask supports many different resource managers.
- We will follow the [Dask documentation](https://distributed.dask.org/en/stable/client.html) regarding the creation of a `Client`.

In [ ]:
from dask.distributed import Client, LocalCluster
cluster = LocalCluster(n_workers=2, threads_per_worker=1, processes=True, memory_limit="2GiB")
client = Client(cluster)

### Create a ROOT dataframe

We now create an RDataFrame based on the same dataset seen in the exercise [rdataframe-dimuon](exercises/rdataframe-dimuon.ipynb).

A Dask `RDataFrame` receives two extra parameters: 
- the executor, which in this case will be the dask client object
- the number of partitions to apply to the dataset (npartitions)

Besides this detail, a Dask `RDataFrame` is not different from a local `RDataFrame`: the analysis presented in this notebook would not change if we wanted to execute it locally.

In [ ]:
# Use a Dask RDataFrame
df = ROOT.RDataFrame("h42",
                "https://root.cern/files/h1big.root",
                executor=client,
                npartitions=4)

### Run your analysis unchanged

- From now on, the rest of your application can be written **exactly** as we have seen with local RDataFrame. 

- The goal of the distributed RDataFrame module is to support all the traditional RDataFrame operations (those that make sense in a distributed context at least). 

- The list of operations that are currently available in distributed mode and can be found in the corresponding [section of the documentation](https://root.cern/doc/master/classROOT_1_1RDataFrame.html#rdf_distrdf)